In [29]:
import regex as re # 注意：必须使用 regex 库以支持 \p{L} 等高级 Unicode 属性
from collections import Counter, defaultdict
from rich import print as pprint
import os
import json
from tqdm import tqdm

In [ ]:
def bbpe(input_path, vocab_size, special_tokens=["<|endoftext|>"]):
    """
    Byte-level BPE 训练函数
    :param input_path: 训练语料 txt 文件路径
    :param vocab_size: 目标词表大小（包含 256 个基础字节和特殊 token）
    :param special_tokens: 特殊 token 列表，如 [<|endoftext|>]
    """
    
    # --- 1. 初始化词表 (0-255 的基础字节) ---
    # 词表初始包含 256 个单字节 token，确保任何文本都能被表示，无 OOV 问题
    vocab = {idx: bytes([idx]) for idx in range(256)}

    # 读取输入文本
    with open(input_path, 'r', encoding='utf-8') as f:
        text = f.read()
    
    # --- 2. 预处理：保护特殊 Token 并进行预分词 ---
    if special_tokens:
        # 将特殊 Token 转义并合并为正则串，用于切割文本（不让 BPE 拆解它们）
        sp_regex = '|'.join(re.escape(token) for token in special_tokens)
        parts = re.split(f'({sp_regex})', text)
        # 仅保留非特殊 Token 的普通文本片段用于 BPE 统计
        train_segments = [s for s in parts if s and s not in special_tokens]
    else:
        train_segments = [text]
    
    # GPT-2 官方预分词正则：将文本预先切分为缩写、单词、数字及标点碎片
    # 目的是防止 BPE 跨越单词边界合并（例如防止把 "apple" 的结尾和 "is" 的开头合并）
    gpt2_pattern = re.compile(r"'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+")
    
    # 统计所有单词碎片的出现频率
    raw_counts = Counter()
    for segment in train_segments:
        words = gpt2_pattern.findall(segment)
        for word in words:
            # word.encode('utf-8') 将字符串转为字节流
            # bytes([b]) 将每个字节整数转为单字节对象，存入元组作为 BPE 的最小单位
            raw_counts[tuple(bytes([b]) for b in word.encode('utf-8'))] += 1
    
    # 将 Counter 展开为列表形式，方便后续利用索引 (idx) 进行操作
    words_list, counts_list = [], []
    for word_tuple, freq in raw_counts.items():
        words_list.append(list(word_tuple)) # 使用 list 方便后续原地执行合并动作
        counts_list.append(freq)
    
    # --- 3. 构造高效统计数据结构 (性能优化的核心) ---
    stats = defaultdict(int)      # 存储相邻字节对 (pair) 在全语料中的总频率
    indices = defaultdict(set)    # 倒排索引：存储 pair 出现在 words_list 中的哪些位置
    
    for idx, word in enumerate(words_list):
        freq = counts_list[idx]
        for i in range(len(word) - 1):
            pair = (word[i], word[i + 1])
            stats[pair] += freq
            indices[pair].add(idx)
    
    # --- 4. 核心合并循环 (Iterative Merging) ---
    # 计算需要执行合并的次数
    num_merges = vocab_size - len(vocab) - len(special_tokens)
    merges = [] # 记录合并规则，如 (b'h', b'e') -> b'he'
    
    pbar = tqdm(range(num_merges), desc="BBPE Training", unit="merge")
    
    for _ in pbar:
        if not stats:
            break
        
        # 寻找当前频率最高且字典序最大的字节对作为合并对象
        best_pair = max(stats.items(), key=lambda x: (x[1], x[0]))[0]
        
        # 如果最高频率已经降为 0，说明没有可以合并的对了，提前结束
        if stats[best_pair] <= 0:
            break
        
        pair_str = (best_pair[0] + best_pair[1]).decode('utf-8', errors='replace')
        pbar.set_postfix({"pair": pair_str, "freq": stats[best_pair]})
        
        merges.append(best_pair)
        new_token = best_pair[0] + best_pair[1] # 拼接成新的子词单位
        
        # 仅针对包含 best_pair 的单词进行更新（利用倒排索引加速，避免全语料扫描）
        relevant_indices = list(indices[best_pair])
        
        for idx in relevant_indices:
            word = words_list[idx]
            freq = counts_list[idx]
            
            i = 0
            while i < len(word) - 1:
                # 寻找匹配 best_pair 的具体位置
                if word[i] == best_pair[0] and word[i + 1] == best_pair[1]:
                    # --- A. 扣除因合并而失效的旧邻居对频率 ---
                    # 1. 处理左侧邻居：(word[i-1], word[i]) 将因为 word[i] 的改变而碎掉
                    if i > 0:
                        prev_pair = (word[i - 1], word[i])
                        stats[prev_pair] -= freq
                        if stats[prev_pair] <= 0: # 及时清理，防止被 max() 误选
                            del stats[prev_pair]
                    
                    # 2. 处理右侧邻居：(word[i+1], word[i+2]) 将因为 word[i+1] 的删除而碎掉
                    if i < len(word) - 2:
                        next_pair = (word[i + 1], word[i + 2])
                        stats[next_pair] -= freq
                        if stats[next_pair] <= 0:
                            del stats[next_pair]
                    
                    # --- B. 执行原地合并 (关键步骤) ---
                    word[i] = new_token
                    del word[i + 1] # 列表长度减 1
                    
                    # --- C. 登记因合并而产生的新邻居对频率 ---
                    # 1. 产生新的左邻居关系：(word[i-1], new_token)
                    if i > 0:
                        new_prev = (word[i - 1], word[i])
                        stats[new_prev] += freq
                        indices[new_prev].add(idx)
                    
                    # 2. 产生新的右邻居关系：(new_token, word[i+1])
                    if i < len(word) - 1:
                        new_next = (word[i], word[i + 1])
                        stats[new_next] += freq
                        indices[new_next].add(idx)
                    
                    # 合并后 i 不自增，原地重新检查新产生的 word[i] 与其右侧的关系
                else:
                    i += 1
        
        # 彻底清理已经处理完的 best_pair 记录
        if best_pair in stats:
            del stats[best_pair]
        if best_pair in indices:
            del indices[best_pair]
    
    # --- 5. 组装最终词表 ---
    # 将合并规则顺序还原为字节内容并加入词表
    for pair in merges:
        new_token = pair[0] + pair[1]
        vocab[len(vocab)] = new_token

    # 最后放入特殊 Token，通常它们会被分配到词表的最后几位
    for token in special_tokens:
        vocab[len(vocab)] = token.encode('utf-8')
    
    return vocab, merges

In [31]:
def bytes_to_unicode():
    """
    创建一个映射表，将 0-255 的每个字节值映射到一个可打印的 Unicode 字符。
    这是 GPT-2 源码中的标准做法，旨在确保字节序列在经过正则处理或打印时不会因为包含不可见字符、空格或控制字符而产生歧义或报错。
    """
    
    # 1. 挑选出原始字节中本身就是“可打印字符”的部分
    # 包括：标准 ASCII 可见字符 (! 到 ~)，以及 Latin-1 字符集中的部分可见符号
    bs = (
        list(range(ord("!"), ord("~") + 1))       # ASCII: 33-126
        + list(range(ord("¡"), ord("¬") + 1))     # Latin-1: 161-172
        + list(range(ord("®"), ord("ÿ") + 1))     # Latin-1: 174-255
    )
    
    # cs 是对应的字符编码集合，初始化为可见字节本身的拷贝
    cs = bs[:]
    
    # n 是一个偏移计数器，用于将“不可见字节”映射到 Unicode 的高位区间
    n = 0
    
    # 2. 遍历 0-255 全部的字节值
    for b in range(256):
        # 如果这个字节不在上面挑选出的“可见列表”中（例如空格、换行、控制字符）
        if b not in bs:
            # 将该“不可见字节”加入字节列表
            bs.append(b)
            # 给它分配一个从 256 开始往后排的 Unicode 码点 (256, 257, 258...)
            # 这样所有的字节都被映射到了一个唯一的、可见的字符位置
            cs.append(256 + n)
            n += 1
            
    # 将所有的码点整数转换为对应的 Unicode 字符
    cs = [chr(n) for n in cs]
    
    # 返回一个字典，键是原始字节 (0-255)，值是对应的可见字符
    # 例如：32 (空格) 会被映射到 'Ġ'，10 (换行) 会被映射到 'Ċ'
    return dict(zip(bs, cs))

In [32]:
def save_vocab(vocab, merges, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    
    byte_encoder = bytes_to_unicode()
    json_vocab = {
        k: "".join(byte_encoder[b] for b in v)
        for k, v in vocab.items()
    }
    with open(os.path.join(output_dir, 'vocab.json'), 'w', encoding='utf-8') as f:
        json.dump(json_vocab, f, indent=2)
        
    with open(os.path.join(output_dir, 'merges.txt'), 'w', encoding='utf-8') as f:
        for p1, p2 in merges:
            s1 = "".join(byte_encoder[b] for b in p1)
            s2 = "".join(byte_encoder[b] for b in p2)
            f.write(f"{s1} + {s2} -> {s1 + s2}\n")

In [33]:
input_path = "data/TinyStoriesV2-GPT4-train.txt"
output_dir = "data/tokenizer_output"

vocab_size = 10000

vocab, merges = bbpe(input_path, vocab_size)
save_vocab(vocab, merges, output_dir)

BBPE Training: 100%|██████████| 9743/9743 [00:33<00:00, 290.06merge/s, pair=bold, freq=447]           
